### Loading the dataset
This is the newly feature engineered dataset from `feature_engineering.ipynb`

In [1]:
import pandas as pd
import numpy as np

# loading the dataset into dataframe
df = pd.read_csv('../data/features.csv')

df.head()

,timestamp,server_id,cpu_percent,memory_percent,disk_io,is_anomaly,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,cpu_percent_roll_mean_10,cpu_percent_roll_standard_deviation_10,...,hour,day_of_week,cpu_percent_rate_of_change,memory_percent_rate_of_change,disk_io_rate_of_change,server_type_batch_worker,server_type_cache,server_type_database,server_type_load_balancer,server_type_web
0,2025-01-01 00:05:00,web_1,19.308678,25.694752,43.981028,0,20.896125,2.244988,20.896125,2.244988,...,0,2,-3.174892,2.419824,-19.470143,0,0,0,0,1
1,2025-01-01 00:10:00,web_1,23.238443,36.885905,57.091144,0,21.676897,2.085378,21.676897,2.085378,...,0,2,3.929764,11.191153,13.110116,0,0,0,0,1
2,2025-01-01 00:15:00,web_1,27.615149,35.100183,51.622232,0,23.161460,3.422705,23.161460,3.422705,...,0,2,4.376707,-1.785722,-5.468912,0,0,0,0,1
3,2025-01-01 00:20:00,web_1,18.829233,28.184342,33.647097,0,22.295015,3.541161,22.295015,3.541161,...,0,2,-8.785916,-6.915841,-17.975135,0,0,0,0,1
4,2025-01-01 00:25:00,web_1,18.829315,34.272010,40.813835,0,21.564164,3.855648,21.717398,3.468963,...,0,2,0.000082,6.087668,7.166738,0,0,0,0,1


### Dropping columns 
Dropping the following:
- `timestamp`: raw datetime, not usuable
- `server_id`: just use the lable or identifier

In [2]:
df = df.drop(columns=["timestamp","server_id"])
print(df.columns)

Index(['cpu_percent', 'memory_percent', 'disk_io', 'is_anomaly',
       'cpu_percent_roll_mean_5', 'cpu_percent_roll_standard_deviation_5',
       'cpu_percent_roll_mean_10', 'cpu_percent_roll_standard_deviation_10',
       'cpu_percent_roll_mean_30', 'cpu_percent_roll_standard_deviation_30',
       'memory_percent_roll_mean_5',
       'memory_percent_roll_standard_deviation_5',
       'memory_percent_roll_mean_10',
       'memory_percent_roll_standard_deviation_10',
       'memory_percent_roll_mean_30',
       'memory_percent_roll_standard_deviation_30', 'disk_io_roll_mean_5',
       'disk_io_roll_standard_deviation_5', 'disk_io_roll_mean_10',
       'disk_io_roll_standard_deviation_10', 'disk_io_roll_mean_30',
       'disk_io_roll_standard_deviation_30', 'cpu_percent_zscore_5',
       'cpu_percent_zscore_10', 'cpu_percent_zscore_30',
       'memory_percent_zscore_5', 'memory_percent_zscore_10',
       'memory_percent_zscore_30', 'disk_io_zscore_5', 'disk_io_zscore_10',
       'disk_i

### Training the model using Isolation forest

In [3]:
y = df['is_anomaly']
# `is_anomaly`: this is the ground truth label, and shoulnd't be used during training
X = df.drop(columns=["is_anomaly"])

print(X.shape)
print(y.value_counts())


(90705, 40)
is_anomaly
0    90536
1      169
Name: count, dtype: int64


In [4]:
# calculate the contamination rate 
contamination_rate = y.sum() / len(y)
print(contamination_rate)

0.0018631828454881208


In [5]:
# fit the model with the dataframe
from sklearn.ensemble import IsolationForest
clf = IsolationForest(contamination=contamination_rate, random_state=42).fit(X)

In [6]:
predictions = clf.predict(X)
print(pd.Series(predictions).value_counts())

 1    90536
-1      169
Name: count, dtype: int64


### Classification report 
`Precision` - How many of the true positves flagged by the model is accurate 
$$Precision = \frac{\text{True Positives (TP)}}{\text{True Positives (TP)} + \text{False Positives (FP)}}$$
`Recall` - How many actually true positives did the model catch 
$$Recall = \frac{\text{True Positives (TP)}}{\text{True Positives (TP)} + \text{False Negatives (FN)}}$$
`F1-score1` - This is the harmonic mean of the recall and precison
$$F_1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

The precison, recall and f1-score (for this first round of training the model) for class `1` is extremley low, meaning the model is not catching anything almost correctly.

In [9]:
# check the models predictions
from sklearn.metrics import classification_report

# if predictions == -1, this would output True
# .astype will convert True to 1
predictions_binary = (predictions == -1).astype(int) # converts -1 to 1

# classification report on the predicitons
classification_report = classification_report(y, predictions_binary)
print(classification_report)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     90536
           1       0.01      0.01      0.01       169

    accuracy                           1.00     90705
   macro avg       0.50      0.50      0.50     90705
weighted avg       1.00      1.00      1.00     90705

